## 1. 평가 환경 및 데이터셋 구성

1. 필수 라이브러리 불러오기

In [2]:
from pathlib import Path

import torch

from anomalib.data import MVTecAD
from anomalib.models import Patchcore
from anomalib.engine import Engine

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention


2. GPU 환경 확인

In [3]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


3. MVTec AD 평가 데이터 구성

In [4]:
datamodule = MVTecAD(
    root="./datasets/MVTecAD",
    category="bottle",
    train_batch_size=32,
    eval_batch_size=32,
)

datamodule.prepare_data()
datamodule.setup()

4. 테스트 데이터 확인

In [5]:
test_loader = datamodule.test_dataloader()

print("Test samples:", len(test_loader.dataset))
print("Test batches:", len(test_loader))

Test samples: 83
Test batches: 3


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 2. 학습된 PatchCore 모델 불러오기

1. PatchCore 구조 재생성

In [6]:
pre_processor = Patchcore.configure_pre_processor(
    image_size=(256, 256),
    center_crop_size=(256, 256),
)

model = Patchcore(
    backbone="wide_resnet50_2",
    layers=["layer2", "layer3"],
    pre_trained=True,
    coreset_sampling_ratio=0.1,
    num_neighbors=9,
    pre_processor=pre_processor,
)

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'pre_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pre_processor'])`.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


2. 평가 Engine 생성

In [7]:
engine = Engine(
    accelerator="gpu",
    devices=1,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
)

3. 저장된 Checkpoint 경로 확인

In [8]:
checkpoint_path = Path("./models/patchcore_bottle.ckpt")

print("Checkpoint exists:", checkpoint_path.exists())

Checkpoint exists: True
